In [ ]:
"""

This notebook has for objective to load any config and then make several tests on it, as generating images, interpolating, computing cosine similarity

"""

In [ ]:
"""
Reusing fonctions of pivotal_tuning notebook

"""

'\nRéutilisation de fonctions du code de pivotal_tuning\n\n'

In [ ]:
import json
import math
from itertools import groupby
from typing import Callable, Dict, List, Optional, Set, Tuple, Type, Union

import numpy as np
import PIL
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from safetensors.torch import safe_open
    from safetensors.torch import save_file as safe_save

    safetensors_available = True
except ImportError:
    from .safe_open import safe_open
import argparse
import hashlib
import inspect
import itertools
import math
import os
import random
import re
from pathlib import Path
from typing import Optional, List, Literal

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.checkpoint
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from huggingface_hub import HfFolder, Repository, whoami
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer
import wandb
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

from PIL import Image
from torch import zeros_like
from torch.utils.data import Dataset
from torchvision import transforms
import glob
import matplotlib.pyplot as plt
import os
import zipfile
from google.colab import files

In [ ]:
"""
We define the modules to be replaced in the UNET and the text encoder, as well as a flag that will be useful later to identify the text encoder embeddings
in the metadata.

"""

UNET_TARGET_REPLACE = {"Transformer2DModel"}

TEXT_ENCODER_TARGET_REPLACE = {"CLIPTextTransformer"}

EMBED_FLAG = "<embed>"

In [ ]:
def _find_modules(
    model,
    ancestor_class: Optional[Set[str]] = None,
    search_class: List[Type[nn.Module]] = [nn.Linear],
    exclude_children_of: Optional[List[Type[nn.Module]]] = [
    ],
    tracked_lora_paths: Optional[Dict[str,List[str]]] = None,
    tracked_lora_weights : Optional[Dict[str,List[float]]] = None,
    tracked_lora_grads : Optional[Dict[str,List[float]]] = None
):
    """
    Find all modules of a certain class (or union of classes) that are direct or
    indirect descendants of other modules of a certain class (or union of classes).

    Returns all matching modules, along with the parent of those modules and the
    names they are referenced by.
    """

    # Get the targets we should replace all linears under
    if ancestor_class is not None:
        ancestors = [
            [fullname, module]
            for fullname, module in model.named_modules()
            if module.__class__.__name__ in ancestor_class
        ]
    else:
        # this, incase you want to naively iterate over all modules.
        ancestors = [module for module in model.modules()]

    # For each target find every linear_class module that isn't a child of a LoraInjectedLinear
    for ancestor_fullname,ancestor in ancestors:
        #.named_modules() donne le chemin entier de l'ancestor au module
        for fullname, module in ancestor.named_modules():
            if any([isinstance(module, _class) for _class in search_class]):
                # Find the direct parent if this is a descendant, not a child, of target
                #"encoder.layer.2.attention.linear" devient path = ['encoder', 'layer', '2', 'attention'] et name = 'linear'
                *path, name = fullname.split(".")
                parent = ancestor
                while path:
                    #path.pop(0) = encoder
                    #parent.get_submodule(path.pop(0)) = parent.encoder
                    #au final : parent.encoder.layer.2.attention
                    parent = parent.get_submodule(path.pop(0))
                # Pour ne pas injecter des loras dans des loras si par exemple on exécute plusieurs fois train
                if exclude_children_of and any(
                    [isinstance(parent, _class) for _class in exclude_children_of]
                ):
                    continue
                #on met à jour les listes qui stockent les valeurs de l'entrainement, en utilisant le chemin complet jusqu'au module où le lora
                #va être injecté
                full_path = f"{model.__class__.__name__}.{ancestor_fullname}.{fullname}" if ancestor_fullname else fullname
                if tracked_lora_paths is not None:
                  tracked_lora_paths[model.__class__.__name__].append(full_path)
                  tracked_lora_weights[full_path] = []
                  tracked_lora_grads[full_path] = []
                # Otherwise, yield it
                yield full_path, parent, name, module

In [ ]:
"""
Roadmap for loading loras from safetensor

1°) parse_safeloras:
    Rebuilds a usable loras dictionary of weights and ranks from the safetensor containing the weights and metadata dictionaries.

2°) apply_lora_on_model:
    Adds the weights of the loras from the loras list to all modules of the model.

3°) apply_lora_from_safetensor:
    Builds the loras dictionary from the safetensor with parse_safeloras, applies this dictionary to the UNET and text_encoder with
    apply_lora_on_model.
"""

"\nRoadmap pour charger les loras à partir du safetensor\n\n1°) parse_safeloras :\n    Reconstruit un dictionnaire loras de poids et de rangs utilisable à partir du safetensor contenant les dictionnaires weights et metadata.\n\n2°) apply_lora_on_model :\n    Ajoute le poids des loras de la liste loras à tous les modules du modèle\n\n3°) apply_lora_from_safetensor :\n    construit le dictionnaire loras à partir du safetensor avec parse_safeloras, applique ce dictionnaire à l'unet et au text_encoder avec\n    apply_lora_on_model\n"

In [ ]:
def parse_safeloras(
    safeloras,
) -> Dict[str, Tuple[List[nn.parameter.Parameter], List[int], List[str]]]:
    """
    Rebuilds a usable loras dictionary of weights and ranks
    from the safetensor containing the weights and metadata dictionaries.
    The loras dictionary is as follows:
    - lora[model] where model is either “unet” or “text_encoder,” is a tuple (weights,ranks,target)
    - weights is the list of up and down weights of all the model's loras
    - ranks are the ranks of the loras
    - target are the targeted modules

    """
    loras = {}
    #safeloras.metadata() récupère le dictionnaire metadate et safeloras.keys() récupère les clés du dictionnaire weights uniquement
    metadata = safeloras.metadata()
    #donne le modèle de la clé ex : "unet:1:up" -> "unet"
    get_name = lambda k: k.split(":")[0]

    keys = list(safeloras.keys())
    #le tri se fait sur le modèle : unet, text_encoder ou les tokens comme <tok1>,<tok2>. Ici on a ajouté toutes les clés unet
    #puis toutes les clés text_encoder dans extract_lora_as_tensor donc ce tri n'est pas forcément utile
    keys.sort(key=get_name)

    for name, module_keys in groupby(keys, get_name):
        info = metadata.get(name)

        if not info:
            raise ValueError(
                f"Tensor {name} has no metadata - is this a Lora safetensor?"
            )

        # Skip Textual Inversion embeds
        if info == EMBED_FLAG:
            continue

        # Handle Loras
        # Extract the targets
        target = json.loads(info)

        # Build the result lists - Python needs us to preallocate lists to insert into them
        module_keys = list(module_keys)
        ranks = [4] * (len(module_keys) // 2)
        weights = [None] * len(module_keys)

        for key in module_keys:
            # Split the model name and index out of the key
            _, idx, direction = key.split(":")
            idx = int(idx)

            # Add the rank
            ranks[idx] = int(metadata[f"{name}:{idx}:rank"])

            # Insert the weight into the list
            idx = idx * 2 + (1 if direction == "down" else 0)
            weights[idx] = nn.parameter.Parameter(safeloras.get_tensor(key))

        loras[name] = (weights, ranks, target)

    return loras

In [ ]:
def apply_lora_on_model(
    model,
    loras,
    target_replace_module,
    r: Union[int, List[int]] = 4,
    delta_weights : Dict[str, float] = {},
):
    """
    Adds the weight of the loras from the loras list to all modules in the model.

    """

    alpha = 1.0

    for full_path,_module, name, _child_module in _find_modules(
        model,
        target_replace_module,
        search_class=[nn.Linear, nn.Conv2d,]
    ):
        with torch.no_grad() :
          if (_child_module.__class__ == nn.Linear) :
              #La matrice de poids up ou down doit être de dimension 2 pour un linear
              if len(loras[0].shape) != 2:
                  continue
              up_weight = loras.pop(0)
              down_weight = loras.pop(0)
              delta = alpha*(up_weight @ down_weight)
              start_weight = _child_module.weight.detach().clone()
              _child_module.weight += delta.type(_child_module.weight.dtype).to(_child_module.weight.device)
              delta_weights[full_path] = torch.norm((_child_module.weight - start_weight),p='fro').item()
          elif (_child_module.__class__ == nn.Conv2d):
              #La matrice de poids up ou down doit être de dimension 4 pour un conv2d (out_channels, in_channels, kernel_height, kernel_width)
              #on a un filtre différent pour chaque duo out_channel,in_channel
              if len(loras[0].shape) != 4:
                  continue
              up_weight = loras.pop(0)
              down_weight = loras.pop(0)
              delta = alpha*(up_weight.flatten(start_dim=1) @ down_weight.flatten(start_dim=1)).reshape(_child_module.weight.data.shape)
              start_weight = _child_module.weight.detach().clone()
              _child_module.weight += delta.type(_child_module.weight.dtype).to(_child_module.weight.device)
              delta_weights[full_path] = torch.norm((_child_module.weight - start_weight),p='fro').item()

In [ ]:
def apply_lora_from_safetensor(pipe, safeloras):
    """
    Applies all safetensor loras to unet and text_encoder

    """
    loras = parse_safeloras(safeloras)
    #Dictionnary to store the norm of the differences of weight between the original model
    #and after loras are loaded, to ensure it worked
    delta_weights = {}

    for name, (lora, ranks, target) in loras.items():
        model = getattr(pipe, name, None)

        if not model:
            print(f"No model provided for {name}, contained in Lora")
            continue

        apply_lora_on_model(model, lora, target, ranks,delta_weights)
    return delta_weights

In [ ]:
"""
Roadmap for loading embeddings from safetensor

1°) parse_safeloras_embeds:
    Rebuilds a usable embeddings dictionary from safetensor containing the weights and metadata dictionaries.

2°) apply_learned_embed_in_clip:
    Uses the embeddings dictionary of new tokens to update their embedding.
"""

"\nRoadmap pour charger les embeddings à partir du safetensor\n\n1°) parse_safeloras_embeds :\n    Reconstruit un dictionnaire embeds d'embeddings utilisable à partir du safetensor contenant les dictionnaires weights et metadata.\n\n2°) apply_learned_embed_in_clip :\n    Utilise le dictionnaire embeds des nouveaux tokens pour mettre a jour leur embedding\n"

In [ ]:
def parse_safeloras_embeds(
    safeloras,
) -> Dict[str, torch.Tensor]:
    """
    Rebuilds a usable embeds dictionary of embeddings
    from safetensor with the weights and metadata dictionaries.
    The embeds dictionary is such that:
    - embeds[token] = embedding of the token

    """
    embeds = {}
    metadata = safeloras.metadata()

    for key in safeloras.keys():
        # Only handle Textual Inversion embeds
        meta = metadata.get(key)
        if not meta or meta != EMBED_FLAG:
            continue

        embeds[key] = safeloras.get_tensor(key)

    return embeds

In [ ]:
def apply_learned_embed_in_clip(
    learned_embeds,
    text_encoder,
    tokenizer,
    idempotent=False,
):

    """
    Add the tokens that are the keys of learned_embeds to the tokenizer, create their IDs, and add the embedding to the text_encoder.

    Parameters:
        idempotent:
            If the token already exists in the tokenizer, we will try to add it in another form, e.g.:
            “mytoken”, “mytoke-1>”, then “mytoke-2>”

    """

    trained_tokens = list(learned_embeds.keys())

    for token in trained_tokens:
        embeds = learned_embeds[token]

        # cast to dtype of text_encoder
        dtype = text_encoder.get_input_embeddings().weight.dtype
        num_added_tokens = tokenizer.add_tokens(token)

        i = 1
        if not idempotent:
            while num_added_tokens == 0:
                print(f"The tokenizer already contains the token {token}.")
                token = f"{token[:-1]}-{i}>"
                print(f"Attempting to add the token {token}.")
                num_added_tokens = tokenizer.add_tokens(token)
                i += 1
        elif num_added_tokens == 0 and idempotent:
            print(f"The tokenizer already contains the token {token}.")
            print(f"Replacing {token} embedding.")
        else :
          print(f"Ajout du token {token}")

        # resize the token embeddings
        text_encoder.resize_token_embeddings(len(tokenizer))

        # get the id for the token and assign the embeds
        token_id = tokenizer.convert_tokens_to_ids(token)
        text_encoder.get_input_embeddings().weight.data[token_id] = embeds
    return token

In [ ]:
def patch_pipe(
    pipe,
    maybe_unet_path,
    token: Optional[str] = None,
    r: int = 4,
    patch_unet=False,
    patch_text=False,
    patch_ti=True,
    idempotent_token=True,
    unet_target_replace_module=UNET_TARGET_REPLACE,
    text_target_replace_module=TEXT_ENCODER_TARGET_REPLACE,
):

    """
    Applies the loras to UNET and the text_encoder, as well as the embeddings of the new tokens to the tokenizer.

    """

    #framework="pt"	on veut accéder aux tensors au format PyTorch (torch.Tensor)
    safeloras = safe_open(maybe_unet_path, framework="pt", device="cpu")
    if patch_unet :
      delta_weights = apply_lora_from_safetensor(pipe, safeloras)
    if patch_ti:
        tok_dict = parse_safeloras_embeds(safeloras)
        apply_learned_embed_in_clip(
            tok_dict,
            pipe.text_encoder,
            pipe.tokenizer,
            idempotent=idempotent_token,
        )
        delta_weights = []
    return delta_weights

In [ ]:
"""
Loading the model and patch with safetensor

"""

'\nChargement du modèle et patch avec le safetensor\n\n'

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "sd-legacy/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16,safety_checker=None)
pipe = pipe.to("cuda")


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
delta_weights = patch_pipe(
    pipe=pipe,
    maybe_unet_path="./tuning_999_0.0001_ (1).safetensors",
    r = 4,
    patch_unet=True,
    patch_text=False,
    patch_ti=False,
    idempotent_token=True,
    unet_target_replace_module=UNET_TARGET_REPLACE,
    text_target_replace_module=TEXT_ENCODER_TARGET_REPLACE,
)

In [ ]:
"""
Verifying that tokens have been added

"""

In [ ]:
def mesure_dist_tokens (text_encoder,tokenizer,placeholder_tokens,initializer_tokens) :
      if len(placeholder_tokens) == 0:
        placeholder_tokens = []
        print("PTI : Placeholder Tokens not given, using null token")
      else:
          placeholder_tokens = placeholder_tokens.split("|")

          assert (
              sorted(placeholder_tokens) == placeholder_tokens
          ), f"Placeholder tokens should be sorted. Use something like {'|'.join(sorted(placeholder_tokens))}'"

      if initializer_tokens is None:
          print("PTI : Initializer Tokens not given, doing random inits")
          initializer_tokens = ["<rand-0.017>"] * len(placeholder_tokens)
      else:
          initializer_tokens = initializer_tokens.split("|")

      assert len(initializer_tokens) == len(
          placeholder_tokens
      ), "Unequal Initializer token for Placeholder tokens."


      print("PTI : Placeholder Tokens", placeholder_tokens)
      print("PTI : Initializer Tokens", initializer_tokens)
      for token,target in zip(placeholder_tokens,initializer_tokens) :
        print("token %s avec pour origine %s" % (token,target))
        i = tokenizer.convert_tokens_to_ids(token)
        j = tokenizer.convert_tokens_to_ids(target)
        vec_i = text_encoder.get_input_embeddings().weight[i]
        vec_j = text_encoder.get_input_embeddings().weight[j]
        dist = torch.norm(vec_i - vec_j, p=2).item()
        print("dist %s"% dist)
        sim = cos_abs = torch.nn.functional.cosine_similarity(vec_i.unsqueeze(0),vec_j.unsqueeze(0)).item()
        print("sim %s"% sim)


In [ ]:
mesure_dist_tokens(pipe.text_encoder,pipe.tokenizer,"<tok1>","character")

In [ ]:
"""
Verification that the loras have been loaded correctly

"""

In [ ]:
import matplotlib.pyplot as plt

# Exemple avec dictionnaire trié
sorted_items = sorted(delta_weights.items(), key=lambda x: x[1], reverse=True)
paths, norm_diffs = zip(*sorted_items)

plt.figure(figsize=(8, len(paths) * 0.3))
bars = plt.barh(paths, norm_diffs, color='skyblue')
plt.xlabel("Différence de norme (frobenius)")
plt.title("Impact de LoRA sur les poids")
plt.tight_layout()
plt.grid(axis='x')

# Affichage des valeurs sur chaque barre
for bar, value in zip(bars, norm_diffs):
    plt.text(
        bar.get_width() + 0.01,                 # position x : un peu après la fin de la barre
        bar.get_y() + bar.get_height() / 2,     # position y : centré sur la barre
        f"{value:.4f}",                         # format de la valeur
        va='center',                            # alignement vertical centré
        fontsize=8,
        color='black'
    )

plt.show()


In [ ]:
"""
Generation of a single image

"""

In [ ]:
prompt = "an anime illustration of <tok1> woman with long blue hair"
image = pipe(prompt).images[0]
image.save(f"./{prompt}.png")

In [ ]:
"""

vSLERP

"""

In [ ]:
def vSLERP(vec1, vec2, t, mean_vec=None):
    """
    vec1, vec2 : torch.Tensor (dimension d)
    t : float in [0,1]
    mean_vec : torch.Tensor (vecteur moyen des embeddings, sinon calculé sur la volée)
    """

    # Recentrage

    if mean_vec is not None:
        vec1 = vec1 - mean_vec
        vec2 = vec2 - mean_vec
    else :
        v1 = vec1
        v2 = vec2

    # Angle entre les deux
    dot = torch.clamp(torch.dot(v1/v1.norm(), v2/v2.norm()), -1.0, 1.0)
    theta = torch.acos(dot)

    sin_theta = torch.sin(theta)
    interp = (torch.sin((1 - t) * theta) / sin_theta) * v1 + (torch.sin(t * theta) / sin_theta) * v2

    if mean_vec is not None:
      result = interp + mean_vec
    else :
      result = interp
    return result.half()

In [1]:
"""
Recovery of tok1 and character embeddings

"""

'\nRécupération des embeddings de tok1 et de character\n\n'

In [ ]:
prompt = "an anime illustration of <tok1>"

# Tokenisation
enc = pipe.tokenizer(prompt, padding="max_length",
          max_length=pipe.tokenizer.model_max_length, return_tensors="pt").to("cuda")
tokens = pipe.tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
print(tokens)
#5

with torch.no_grad():
    out = pipe.text_encoder(**enc)
    # Matrice utilisée par le UNet (post-transformer, normalisée par LayerNorm finale)
    prompt_embeds = out.last_hidden_state   # [1, 77, 768]
    vec_tok1 = prompt_embeds[0, 5]


prompt = "an anime illustration of character"

# Tokenisation
enc = pipe.tokenizer(prompt, padding="max_length",
          max_length=pipe.tokenizer.model_max_length, return_tensors="pt").to("cuda")
tokens = pipe.tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
print(tokens)

with torch.no_grad():
    out = pipe.text_encoder(**enc)
    # Matrice utilisée par le UNet (post-transformer, normalisée par LayerNorm finale)
    prompt_embeds = out.last_hidden_state   # [1, 77, 768]
    vec_class = prompt_embeds[0,5]

In [ ]:
"""
Interpolation

"""

In [ ]:
vslerp = vSLERP(vec_tok1, vec_class, 0.5, mean_vec=None)

In [ ]:
"""
Verify that the interpolation was successful; we should have an embedding with a high cosine similarity to tok1 and character.

"""

In [ ]:
print(torch.matmul(vec_tok1/vec_tok1.norm(), vslerp/vslerp.norm()))
print(torch.matmul(vec_class/vec_class.norm(), vslerp/vslerp.norm()))
print(torch.matmul(vec_class/vec_class.norm(), vec_tok1/vec_tok1.norm()))

In [ ]:
"""
Replacing the tok1 embedding with the vslerp embedding

"""

In [ ]:
prompt = "an anime illustration of <tok1> woman with long blue hair"

# Tokenisation
enc = pipe.tokenizer(prompt, return_tensors="pt").to(pipe.device)

with torch.no_grad():
    out = pipe.text_encoder(**enc)
    # Matrice utilisée par le UNet (post-transformer, normalisée par LayerNorm finale)
    prompt_embeds = out.last_hidden_state.float()   # [1, 77, 768]
    prompt_embeds[0, 5] = vslerp.float()

In [ ]:
"""
Generating an image from interpolation to see if we get a character halfway between a random character and tok1.

"""

In [ ]:
prompt = "an anime illustration of <tok1>"
images = pipe(
        prompt
    ).images
images[0].save(f"./{prompt}.png")

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
"""
Calculation of cosine similarity

"""

In [ ]:
"""
Create a list of images in the dataset from the storage path.
"""

instance_data_root = Path("./instance")

instance_images_path = []

#Le résultat est une liste de chemins d'images ["./images/cat1.jpg", "./images/cat2.png", "./images/cat3.jpeg"
#glob.glob() est utilisé pour construire automatiquement une liste de fichiers
#correspondant à un motif de nom de fichier, par exemple toutes les images dans un dossier avec certaines extensions
instance_images_path = (
    glob.glob(str(instance_data_root) + "/*.jpg")
    + glob.glob(str(instance_data_root) + "/*.png")
    + glob.glob(str(instance_data_root) + "/*.jpeg")
)

instance_images = [Image.open(path) for path in instance_images_path]

for i in range (len(instance_images)):
  if not instance_images[i].mode == "RGB":
      instance_images[i] = instance_images[i].convert("RGB")

"""
We load CLIP to calculate cosine similarity.
"""

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")


In [ ]:
"""
Calculation of cosine similarity for vslerp

"""
os.makedirs("generated_images", exist_ok=True)
prompt = "an anime illustration of <tok1>"
mean = 0
for i in range(5) :
  image = pipe(prompt).images[0]
  filepath = f"generated_images/image{i}.png"
  image.save(filepath)
  for j in range(5) :
    slerp = pipe(
        prompt_embeds=prompt_embeds
    ).images
    filepath = f"generated_images/slerp{j}.png"
    slerp.save(filepath)
    # Préparer les images pour CLIP
    inputs = processor(images=[image, slerp], return_tensors="pt")

    # Passer les images dans l'encodeur visuel
    with torch.no_grad():
        image_features = model.get_image_features(**{k: v for k, v in inputs.items() if k.startswith("pixel_values")})

    # Normaliser les embeddings (important pour cosine similarity)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Calculer la similarité cosinus
    cosine_sim = torch.matmul(image_features[0], image_features[1].T)
    mean += cosine_sim.item()
mean = mean/(10*5)
print(mean)
# Création du zip
zip_filename = "images.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for root, dirs, files in os.walk("generated_images"):
        for file in files:
            zipf.write(os.path.join(root, file))

In [ ]:
"""
Calculation of cosine similarity for images

"""

prompt = "an anime illustration of <tok1>"
mean = 0
for i in range(10) :
  image = pipe(prompt).images[0]
  for instance in instance_images :
    # Préparer les images pour CLIP
    inputs = processor(images=[image, instance], return_tensors="pt")

    # Passer les images dans l'encodeur visuel
    with torch.no_grad():
        image_features = model.get_image_features(**{k: v for k, v in inputs.items() if k.startswith("pixel_values")})

    # Normaliser les embeddings (important pour cosine similarity)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Calculer la similarité cosinus
    cosine_sim = torch.matmul(image_features[0], image_features[1].T)
    mean += cosine_sim.item()
mean = mean/(10*5)
print(mean)

In [ ]:
"""
Calculation of cosine similarity for prompts
"""

# prompts_list = [("an anime illustration of <tok1> in the kitchen","an anime illustration of in the kitchen"),
#                 ("an anime illustration of <tok1> cooking pasta","an anime illustration of cooking pasta"),
#                 ("an anime illustration of <tok1> cooking pasta in the kitchen","an anime illustration of cooking pasta in the kitchen")
#                 ]
prompts_list = [("an anime illustration of <tok1> on his bed","an anime illustration of on his bed"),
                ("an anime illustration of <tok1> reading a book","an anime illustration of reading a book"),
                ("an anime illustration of <tok1> reading a book on his bed","an anime illustration of reading a book on his bed")
                ]
for prompt1,prompt2 in prompts_list :
  # Tokenisation
  inputs = tokenizer([prompt2], padding=True, return_tensors="pt")

  # Encoder le texte
  with torch.no_grad():
      text_features = model.get_text_features(**inputs)

  # Normaliser (L2 norm = 1)
  text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

  for i in range(10) :
    image = pipe(prompt1).images[0]
    # Préparer les images pour CLIP
    inputs = processor(images=[image], return_tensors="pt")

    # Passer les images dans l'encodeur visuel
    with torch.no_grad():
        image_features = model.get_image_features(**{k: v for k, v in inputs.items() if k.startswith("pixel_values")})

    if i == 0 : mean = image_features/10
    else : mean += image_features/10

  mean = mean/mean.norm(p=2, dim=-1, keepdim=True)
  cosine_sim = torch.matmul(mean[0], text_features[0].T).item()
  print(cosine_sim)


